In [2]:
import torch
from transformers import AutoTokenizer, EsmForProteinFolding

device = "cuda:3"


In [ ]:

tokenizer = AutoTokenizer.from_pretrained("facebook/esmfold_v1")

model = EsmForProteinFolding.from_pretrained(
    "facebook/esmfold_v1",
    low_cpu_mem_usage=True,
    cache_dir="/raid/adambiel/models/esmfold"
)

Loading weights: 100%|██████████| 4498/4498 [00:02<00:00, 1964.53it/s]
[transformers] EsmForProteinFolding LOAD REPORT from: facebook/esmfold_v1
Key                                | Status  | 
-----------------------------------+---------+-
esm.contact_head.regression.bias   | MISSING | 
esm.contact_head.regression.weight | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [3]:
model.device

device(type='cpu')

In [4]:
# Reduce memory usage
model.esm = model.esm.half()
model.trunk.set_chunk_size(32)


In [5]:
model = model.to(device)
model.eval()

EsmForProteinFolding(
  (esm): EsmModel(
    (embeddings): EsmEmbeddings(
      (word_embeddings): Embedding(33, 2560, padding_idx=1)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (rotary_embeddings): EsmRotaryEmbedding()
    (encoder): EsmEncoder(
      (layer): ModuleList(
        (0-35): 36 x EsmLayer(
          (attention): EsmAttention(
            (self): EsmSelfAttention(
              (query): Linear(in_features=2560, out_features=2560, bias=True)
              (key): Linear(in_features=2560, out_features=2560, bias=True)
              (value): Linear(in_features=2560, out_features=2560, bias=True)
            )
            (output): EsmSelfOutput(
              (dense): Linear(in_features=2560, out_features=2560, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (LayerNorm): LayerNorm((2560,), eps=1e-05, elementwise_affine=True, bias=True)
          )
          (intermediate): EsmIntermediate(
            (dense): Linear(in_

In [6]:
sequence = "MGAGASAEEKHSRELEKKLKEDAEKDARTVKLLLLGAGESGKSTIVKQMKIIHQDGYSLEECLEFIAIIYGNTLQSILAIVRAMTTLNIQYGDSARQDDARKLMHMADTIEEGTMPKEMSDIIQRLWKDSGIQACFERASEYQLNDSAGYYLSDLERLVTPGYVPTEQDVLRSRVKTTGIIETQFSFKDLNFRMFDVGGQRSERKKWIHCFEGVTCIIFIAALSAYDMVLVEDDEVNRMHESLHLFNSICNHRYFATTSIVLFLNKKDVFFEKIKKAHLSICFPDYDGPNTYEDAGNYIKVQFLELNMRRDVKEIYSHMTCATDTQNVKFVFDAVTDIIIKENLKDCGLF"

inputs = tokenizer(
    [sequence],
    return_tensors="pt",
    add_special_tokens=False,
)

inputs = {k: v.to(device) for k, v in inputs.items()}

In [7]:
len(sequence)

350

In [8]:
with torch.no_grad():
    outputs = model(**inputs)

In [9]:
outputs.aligned_confidence_probs.shape

torch.Size([1, 350, 350, 64])

In [10]:
# Per-residue pLDDT
plddt = outputs.plddt

print("pLDDT shape:", plddt.shape)
print("Mean pLDDT:", plddt.mean().item())

# Predicted TM-score
print("pTM:", outputs.ptm.item())

pLDDT shape: torch.Size([1, 350, 37])
Mean pLDDT: 0.886335551738739
pTM: 0.9486531615257263


In [15]:
plddt

tensor([[[0.8373, 0.8657, 0.8528,  ..., 0.7139, 0.6661, 0.5722],
         [0.8204, 0.8447, 0.8356,  ..., 0.6982, 0.6263, 0.5655],
         [0.8711, 0.8819, 0.8884,  ..., 0.7116, 0.6633, 0.5751],
         ...,
         [0.8868, 0.9052, 0.8869,  ..., 0.7470, 0.8542, 0.5890],
         [0.8707, 0.8715, 0.8100,  ..., 0.7317, 0.8449, 0.5862],
         [0.8497, 0.8526, 0.7574,  ..., 0.7202, 0.7615, 0.5791]]],
       device='cuda:3')

In [11]:
outputs.ptm

tensor(0.9487, device='cuda:3')

In [12]:
pdbs = model.output_to_pdb(outputs)

In [13]:
with open("structure.pdb", "w") as f:
    f.write(pdbs[0])